# EDA Pipeline Runner (Kaggle)

Этот ноутбук клонирует репозиторий, скачивает данные с Mail.ru и запускает пайплайн EDA.


## 0. Клон репозитория и установка зависимостей
Задайте переменную `REPO_URL` на реальный GitHub-репозиторий.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/IceDarold/Vseros_A.git'  # замените при необходимости
REPO_DIR = Path('/kaggle/working/Vseros_A')

if REPO_DIR.exists():
    print('Каталог уже существует, обновляем репозиторий...')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'EDA/requirements.txt')])

print('Готово: репозиторий в', REPO_DIR)


## 1. Переход в рабочий каталог


In [ ]:
import os
os.chdir('/kaggle/working/Vseros_A')
print('Текущий каталог:', os.getcwd())


## 2. Скачивание данных (Mail.ru baseline)


In [ ]:
import os
import re
import tarfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

MAIL_RU_TRAIN = 'https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/train_data.tar'
MAIL_RU_TEST = 'https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar'

workspace = Path('/kaggle/working/eda_workspace')
raw_root = workspace / 'data/raw'
meta_dir = workspace / 'data/meta'
downloads_dir = workspace / 'downloads'

for path in (raw_root, meta_dir, downloads_dir):
    path.mkdir(parents=True, exist_ok=True)

def get_direct_file_link(mailru_file_url: str) -> str:
    resp = requests.get(mailru_file_url)
    resp.raise_for_status()
    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', resp.text)
    if not match:
        raise RuntimeError('Не удалось получить прямую ссылку Mail.ru')
    base_url = match.group(1)
    parts = mailru_file_url.strip('/').split('/')[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"

def download(url: str, dst: Path) -> Path:
    if dst.exists():
        print(dst.name, 'уже скачан')
        return dst
    direct = get_direct_file_link(url)
    print('Скачиваем', url, '→', dst)
    with requests.get(direct, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dst, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=dst.name) as bar:
            for chunk in r.iter_content(8192):
                f.write(chunk)
                bar.update(len(chunk))
    return dst

train_tar = download(MAIL_RU_TRAIN, downloads_dir / 'train_data.tar')
test_tar = download(MAIL_RU_TEST, downloads_dir / 'test_data.tar')

def extract_tar(archive: Path, target: Path) -> None:
    with tarfile.open(archive, 'r:*') as tar:
        def is_within_directory(directory: Path, target_path: Path) -> bool:
            abs_directory = os.path.abspath(directory)
            abs_target = os.path.abspath(target_path)
            return os.path.commonprefix([abs_directory, abs_target]) == abs_directory
        for member in tar.getmembers():
            member_path = target / member.name
            if not is_within_directory(target, member_path):
                raise Exception('Обнаружена попытка выхода за пределы директории при распаковке')
        tar.extractall(target)

extract_tar(train_tar, raw_root)
extract_tar(test_tar, raw_root)

wb_src = raw_root / 'train_opus' / 'word_bounds.json'
if wb_src.exists():
    meta_dir.mkdir(parents=True, exist_ok=True)
    (meta_dir / 'word_bounds.json').write_bytes(wb_src.read_bytes())
else:
    raise FileNotFoundError('Не найден word_bounds.json после распаковки train_data.tar')

print('Данные подготовлены в', workspace)


## 3. Подготовка конфигурации под Kaggle


In [ ]:
from pathlib import Path
import yaml

workspace = Path('/kaggle/working/eda_workspace')
cfg_path = Path('EDA/configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['paths'] = {
    'project_root': '/kaggle/working',
    'data_root': str(workspace / 'data'),
    'raw_train': str(workspace / 'data/raw/train_opus'),
    'raw_test': str(workspace / 'data/raw/test_opus'),
    'meta': str(workspace / 'data/meta'),
    'interim_wav': str(workspace / 'data/interim/wav16k_mono'),
    'tables': str(workspace / 'data/tables'),
    'samples': str(workspace / 'data/samples'),
    'reports': str(workspace / 'reports'),
    'figs': str(workspace / 'reports/figs'),
    'env_log': str(workspace / 'data/meta/env_versions.txt'),
    'readme_short': str(workspace / 'data/meta/README_EDA.md')
}
cfg['downloads']['train']['output'] = str(workspace / 'downloads/train_data.tar')
cfg['downloads']['train']['extract_to'] = str(workspace / 'data/raw')
cfg['downloads']['test']['output'] = str(workspace / 'downloads/test_data.tar')
cfg['downloads']['test']['extract_to'] = str(workspace / 'data/raw')
cfg['dataset']['word_bounds_path'] = str(workspace / 'data/meta/word_bounds.json')
cfg['report']['summary_path'] = str(workspace / 'reports/EDA_Summary.md')
cfg['report']['summary_pdf_path'] = str(workspace / 'reports/EDA_Summary.pdf')

kaggle_cfg = Path('EDA/configs/kaggle.yaml')
kaggle_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print('Конфигурация обновлена:', kaggle_cfg)


## 4. Запуск пайплайна


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Vseros_A
CFG="EDA/configs/kaggle.yaml"
stages=(prepare download inventory labels convert vad negatives duplicates cv windowing slices report)
for stage in "${stages[@]}"; do
  echo "Running stage: ${stage}"
  python -m EDA.run "${stage}" --config="$CFG" --force
  echo ""
done


## 5. Итоговые артефакты


In [ ]:
from pathlib import Path
root = Path('/kaggle/working/eda_workspace')
for path in sorted(root.glob('**/*')):
    if path.is_file():
        print(path.relative_to('/kaggle/working'))
